# Dynamic Society Friction Simulator — Colab Training
## 100 Compute Unit Budget Edition

**Run all cells top to bottom: Runtime → Run all**

| GPU | Hours | Epochs | Units |
|-----|-------|--------|-------|
| L4  | 42    | 9      | 100   |
| A100| 13.5  | 4      | 100   |
| T4  | 60    | 12     | 100   |


## Section 1 — GPU Detection, Drive Mount, Budget Tracker


In [ ]:
!pip install -q psutil pyyaml
import os, time, json, shutil
from datetime import datetime
from pathlib import Path
import torch

# Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_AVAILABLE = True
    print("Drive mounted")
except Exception as e:
    print(f"Drive not available: {e}")
    DRIVE_AVAILABLE = False

# Detect GPU
print("
" + "="*60)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print("="*60)

GPU_TYPE    = "none"
GPU_VRAM_GB = 0.0
if torch.cuda.is_available():
    GPU_NAME    = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    for t in ["A100","H100","L4","V100","T4"]:
        if t in GPU_NAME:
            GPU_TYPE = t
            break
    print(f"GPU : {GPU_NAME}")
    print(f"VRAM: {GPU_VRAM_GB:.1f} GB")
    print(f"Type: {GPU_TYPE}")
else:
    GPU_NAME = "CPU"
    print("No GPU -- enable: Runtime -> Change runtime type -> GPU")
print(f"PyTorch: {torch.__version__}")


In [ ]:
class BudgetTracker:
    PRICING = {"A100":7.35,"H100":15.0,"L4":2.35,"V100":2.70,"T4":1.67}
    def __init__(self, total_budget=100, gpu_type="none"):
        self.total_budget   = total_budget
        self.gpu_type       = gpu_type
        self.start_time     = time.time()
        self.units_per_hour = self.PRICING.get(gpu_type, 5.0)
        self.log_file       = Path("/content/budget_log.json")
        self.log            = json.loads(self.log_file.read_text()) if self.log_file.exists() else {"sessions":[]}
        print(f"BudgetTracker: {gpu_type} @ {self.units_per_hour} units/hr")
        print(f"Budget: {total_budget} units = {total_budget/self.units_per_hour:.1f} hours")
    def elapsed_h(self):   return (time.time()-self.start_time)/3600
    def consumed(self):    return self.elapsed_h()*self.units_per_hour
    def remaining(self):   return max(0, self.total_budget-self.consumed())
    def pct(self):         return (self.consumed()/self.total_budget)*100
    def should_stop(self): return self.pct()>=90
    def status(self):
        icon = "🔴" if self.pct()>=90 else "🟠" if self.pct()>=75 else "🟢"
        print(f"{icon} {self.consumed():.2f}/{self.total_budget} units ({self.pct():.1f}%) | "
              f"{self.remaining():.1f} units remaining ({self.remaining()/self.units_per_hour:.1f}h)")
    def save(self):
        self.log["sessions"].append({"time":datetime.now().isoformat(),"consumed":self.consumed(),"gpu":self.gpu_type})
        self.log_file.write_text(json.dumps(self.log,indent=2))
        if DRIVE_AVAILABLE:
            try: shutil.copy2(self.log_file,"/content/drive/MyDrive/dsfs-budget-log.json")
            except: pass

budget = BudgetTracker(total_budget=100, gpu_type=GPU_TYPE)
budget.status()


## Section 2 — Clone Repo & Install Dependencies


In [ ]:
import os, subprocess

REPO_URL    = "https://github.com/vivek797029/Dynamic-Societal-Friction-Simulator.git"
PROJECT_DIR = "/content/Dynamic-Societal-Friction-Simulator"

if not os.path.exists(PROJECT_DIR):
    print("Cloning repo...")
    subprocess.run(["git","clone",REPO_URL,PROJECT_DIR], check=True)
    print("Cloned")
else:
    print("Repo exists -- pulling latest...")
    subprocess.run(["git","-C",PROJECT_DIR,"pull"], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")


In [ ]:
import os
os.chdir("/content/Dynamic-Societal-Friction-Simulator")
print("Installing project dependencies...")
!pip install -e ".[dev,eval]" -q
if GPU_TYPE in ("A100","H100"):
    print("Installing Flash Attention 2...")
    !pip install flash-attn --no-build-isolation -q
print("All dependencies installed")


## Section 3 — GPU Config & Training Parameters


In [ ]:
import os, yaml
os.chdir("/content/Dynamic-Societal-Friction-Simulator")

with open("configs/model_config.yaml") as f:
    cfg = yaml.safe_load(f)

print(f"GPU: {GPU_TYPE}  VRAM: {GPU_VRAM_GB:.1f} GB")
print("="*60)

if GPU_TYPE == "L4":
    cfg["training"]["num_train_epochs"]            = 9
    cfg["training"]["per_device_train_batch_size"] = 2
    cfg["training"]["gradient_accumulation_steps"] = 16
    cfg["training"]["max_seq_length"]              = 2048
    cfg["training"]["bf16"]                        = True
    cfg["training"]["fp16"]                        = False
    cfg["training"]["save_steps"]                  = 50
    cfg["training"]["eval_steps"]                  = 50
    cfg["lora"]["r"]                               = 64
    cfg["lora"]["lora_alpha"]                      = 128
    print("L4: 9 epochs | batch 2x16=32 | seq 2048 | bf16")
elif GPU_TYPE == "A100":
    cfg["training"]["num_train_epochs"]            = 4
    cfg["training"]["per_device_train_batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["max_seq_length"]              = 3072
    cfg["training"]["bf16"]                        = True
    cfg["training"]["fp16"]                        = False
    cfg["training"]["save_steps"]                  = 50
    cfg["training"]["eval_steps"]                  = 50
    cfg["lora"]["r"]                               = 96
    cfg["lora"]["lora_alpha"]                      = 192
    print("A100: 4 epochs | batch 4x8=32 | seq 3072 | bf16")
else:
    cfg["training"]["num_train_epochs"]            = 12
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 32
    cfg["training"]["max_seq_length"]              = 2048
    cfg["training"]["bf16"]                        = False
    cfg["training"]["fp16"]                        = True
    cfg["training"]["optim"]                       = "paged_adamw_8bit"
    cfg["training"]["save_steps"]                  = 50
    cfg["training"]["eval_steps"]                  = 50
    cfg["lora"]["r"]                               = 48
    cfg["lora"]["lora_alpha"]                      = 96
    print("T4/V100: 12 epochs | batch 1x32=32 | seq 2048 | fp16")

cfg["training"]["save_total_limit"]       = 5
cfg["training"]["load_best_model_at_end"] = True
cfg["training"]["early_stopping"]         = False
cfg["gdrive"]["enabled"]                  = DRIVE_AVAILABLE
cfg["gdrive"]["sync_every_n_steps"]       = 50

with open("configs/model_config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

eff = cfg["training"]["per_device_train_batch_size"] * cfg["training"]["gradient_accumulation_steps"]
print(f"Config saved: epochs={cfg['training']['num_train_epochs']}  eff_batch={eff}  "
      f"lr={cfg['training']['learning_rate']}  lora_r={cfg['lora']['r']}")


## Section 4 — Generate Training Data


In [ ]:
import os
os.chdir("/content/Dynamic-Societal-Friction-Simulator")

if os.path.exists("data/processed/train.jsonl"):
    n = sum(1 for _ in open("data/processed/train.jsonl"))
    print(f"Data already exists: {n:,} samples -- skipping")
else:
    print("Generating 50k training samples (~2-5 min)...")
    !dsfs generate-data --num-samples 50000 --output-dir data/processed --seed 42
    print("Data generation complete")


## Section 5 — Weights & Biases (Optional)


In [ ]:
import os
# Uncomment to enable W&B:
# import wandb; wandb.login()
os.environ["WANDB_DISABLED"] = "true"
print("W&B disabled. Uncomment above to enable.")


## Section 6 — Train!

Auto-resumes from checkpoint on Colab disconnect. Just re-run all.


In [ ]:
import os, logging
os.chdir("/content/Dynamic-Societal-Friction-Simulator")
logging.basicConfig(level=logging.INFO)

from src.model.trainer import train

print("="*70)
print("STARTING TRAINING")
print("="*70)
budget.status()

try:
    train(config_path="configs/model_config.yaml")
    budget.save()
    budget.status()
    print("TRAINING COMPLETE!")
except KeyboardInterrupt:
    print("Interrupted")
    budget.save()
except Exception as e:
    print(f"Error: {e}")
    budget.save()
    raise


## Section 7 — Budget Status


In [ ]:
budget.status()


## Section 8 — Evaluate


In [ ]:
import os
from pathlib import Path
os.chdir("/content/Dynamic-Societal-Friction-Simulator")

from src.model.inference import FrictionLLM

adapter = Path("./outputs/checkpoints/final_adapter")
if adapter.exists():
    llm = FrictionLLM(config_path="configs/model_config.yaml", adapter_path=str(adapter))
    scenario = ("A city with tensions between residents and immigrants. "
                "A populist politician blames immigrants for housing costs. "
                "Youth activists organize counter-protests.")
    print("Scenario:", scenario)
    print("
Output:")
    print(llm.analyze_friction(scenario))
else:
    print(f"Adapter not found at {adapter.absolute()} -- training may not have completed")


## Section 9 — Export to Google Drive


In [ ]:
import os, shutil, json
from pathlib import Path
os.chdir("/content/Dynamic-Societal-Friction-Simulator")

adapter = Path("./outputs/checkpoints/final_adapter")
if not adapter.exists():
    print("No adapter found -- run training first")
else:
    size_mb = sum(f.stat().st_size for f in adapter.rglob("*") if f.is_file())/(1024**2)
    print(f"Adapter: {size_mb:.1f} MB")
    if DRIVE_AVAILABLE:
        dst = Path("/content/drive/MyDrive/dsfs-checkpoints/final_adapter")
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(adapter, dst)
        print(f"Saved to Drive: {dst}")
    summary = {
        "model":"Mistral-7B-Instruct-v0.3","adapter":"QLoRA",
        "lora_r":cfg["lora"]["r"],"epochs":cfg["training"]["num_train_epochs"],
        "size_mb":round(size_mb,1),"gpu":GPU_TYPE,
        "budget_used":round(budget.consumed(),2),"budget_total":budget.total_budget
    }
    Path("./outputs").mkdir(parents=True, exist_ok=True)
    Path("./outputs/model_summary.json").write_text(json.dumps(summary,indent=2))
    budget.save()
    print("Summary:", summary)
